In [8]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Embedding, LSTM
import numpy as np
import os
import time

In [9]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print("Device details: ", tf.config.list_physical_devices('GPU'))

Num GPUs Available:  1
Device details:  [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [10]:
# ==========================================
# 1. HYPERPARAMETERS & CONFIGURATION
# ==========================================
BATCH_SIZE = 64
BUFFER_SIZE = 1000
EMBEDDING_DIM = 256
UNITS = 512
VOCAB_SIZE = 5000 # Limit vocab to top 5000 words
MAX_LENGTH = 35   # Max words per caption

# ==========================================
# 2. DATA PREPARATION (Flickr8k)
# ==========================================
# Assuming captions.txt has format: image_name.jpg,caption text
image_folder = "Images/"
caption_file = "captions.txt"

# Read file and pair image paths with captions
image_paths = []
captions = []

with open(caption_file, 'r') as f:
    next(f) # Skip header if there is one
    for line in f:
        img_name, caption = line.strip().split(',', 1)
        image_paths.append(os.path.join(image_folder, img_name))
        captions.append(f"<start> {caption} <end>")

# Tokenization
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=VOCAB_SIZE, 
                                                  oov_token="<unk>", 
                                                  filters='!"#$%&()*+.,-/:;=?@[\]^_`{|}~ ')
tokenizer.fit_on_texts(captions)
tokenizer.word_index['<pad>'] = 0
tokenizer.index_word[0] = '<pad>'

# Convert text to sequences of integers and pad them
train_seqs = tokenizer.texts_to_sequences(captions)
cap_vector = tf.keras.preprocessing.sequence.pad_sequences(train_seqs, maxlen=MAX_LENGTH, padding='post')

# Image loading and preprocessing function
def load_image(image_path, cap):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (224, 224))
    img = tf.keras.applications.resnet50.preprocess_input(img)
    return img, cap

# Create tf.data.Dataset
dataset = tf.data.Dataset.from_tensor_slices((image_paths, cap_vector))
dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(buffer_size=tf.data.AUTOTUNE)


In [11]:


class ImageEncoder(tf.keras.Model):
    def __init__(self, embedding_dim):
        super(ImageEncoder, self).__init__()
        
        # 1. Load pre-trained ResNet50, minus the final classification layers
        self.resnet = ResNet50(include_top=False, weights='imagenet')
        
        # 2. Freeze the ResNet weights (Best practice for early training)
        self.resnet.trainable = False
        
        # 3. Dense layer to map ResNet's output size to your embedding dimension
        self.fc = Dense(embedding_dim)

    def call(self, x):
        # Expected input 'x' shape: (batch_size, 224, 224, 3) for ResNet50
        x = self.resnet(x)
        
        # ResNet50 raw output shape: (batch_size, 7, 7, 2048)
        # We need to flatten the 7x7 spatial grid into a sequence of 49 "pixels/regions"
        # Reshaped shape: (batch_size, 49, 2048)
        x = tf.reshape(x, (tf.shape(x)[0], -1, x.shape[3]))
        
        # Pass through dense layer to match dimensions for Attention
        # Final shape: (batch_size, 49, embedding_dim)
        x = self.fc(x)
        x = tf.nn.relu(x)
        
        return x

class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = Dense(units)
        self.W2 = Dense(units)
        self.V = Dense(1)
    
    def call(self, query, values):
        query_with_time_axis = tf.expand_dims(query, 1)
        score = self.V(tf.nn.tanh(self.W1(query_with_time_axis) + self.W2(values)))
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, dec_units):
        super(Decoder, self).__init__()
        self.dec_units = dec_units
        self.embedding = Embedding(vocab_size, embedding_dim)
        
        self.lstm = LSTM(self.dec_units,
                         return_sequences=True,
                         return_state=True)
        self.fc = Dense(vocab_size)
        self.attention = BahdanauAttention(self.dec_units)

    def call(self, x, hidden, enc_output):
        context_vector, attention_weights = self.attention(hidden, enc_output)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)
        output, state_h, state_c = self.lstm(x)
        output = tf.reshape(output, (-1, output.shape[2]))
        x = self.fc(output)
        return x, state_h, state_c, attention_weights

In [12]:
# ==========================================
# 3. MODEL ARCHITECTURE
# ==========================================
# (Insert your ImageEncoder, BahdanauAttention, and Decoder classes here)

encoder = ImageEncoder(EMBEDDING_DIM)
decoder = Decoder(VOCAB_SIZE, EMBEDDING_DIM, UNITS)

# ==========================================
# 4. TRAINING SETUP & CHECKPOINTS (SAVING)
# ==========================================
optimizer = tf.keras.optimizers.Adam()
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    return tf.reduce_mean(loss_)

# CHECKPOINT MANAGER - This is how you save custom models!
checkpoint_path = "./checkpoints/train"
ckpt = tf.train.Checkpoint(encoder=encoder,
                           decoder=decoder,
                           optimizer=optimizer)
# Keep only the 5 most recent checkpoints to save disk space
ckpt_manager = tf.train.CheckpointManager(ckpt, checkpoint_path, max_to_keep=5)

# If a checkpoint exists, restore the latest one.
if ckpt_manager.latest_checkpoint:
    ckpt.restore(ckpt_manager.latest_checkpoint)
    print("Latest checkpoint restored!")

# ==========================================
# 5. THE TRAINING LOOP
# ==========================================
@tf.function
def train_step(img_tensor, target):
    loss = 0
    # Initialize the decoder input with <start> tokens
    dec_input = tf.expand_dims([tokenizer.word_index['<start>']] * target.shape[0], 1)

    with tf.GradientTape() as tape:
        features = encoder(img_tensor)
        hidden = tf.zeros((target.shape[0], UNITS))

        for i in range(1, target.shape[1]):
            # passing the features through the decoder
            predictions, hidden, _, _ = decoder(dec_input, hidden, features)
            loss += loss_function(target[:, i], predictions)
            
            # Teacher forcing
            dec_input = tf.expand_dims(target[:, i], 1)

    total_loss = (loss / int(target.shape[1]))
    trainable_variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, trainable_variables)
    optimizer.apply_gradients(zip(gradients, trainable_variables))

    return loss, total_loss

Latest checkpoint restored!


In [13]:
EPOCHS = 20

for epoch in range(EPOCHS):
    start = time.time()
    total_loss = 0

    for (batch, (img_tensor, target)) in enumerate(dataset):
        batch_loss, t_loss = train_step(img_tensor, target)
        total_loss += t_loss

        if batch % 100 == 0:
            print(f'Epoch {epoch+1} Batch {batch} Loss {t_loss.numpy():.4f}')
            
    # Save the model every epoch
    ckpt_save_path = ckpt_manager.save()
    print(f'Saving checkpoint for epoch {epoch+1} at {ckpt_save_path}')
    print(f'Time taken for 1 epoch: {time.time() - start:.2f} sec\n')

/home/pinaka-linux/miniconda3/envs/tf_env/lib/python3.10/site-packages/keras/src/optimizers/base_optimizer.py:870: UserWarning: Gradients do not exist for variables ['image_encoder_1/dense_5/kernel', 'decoder_1/bahdanau_attention_1/dense_7/kernel', 'decoder_1/bahdanau_attention_1/dense_7/kernel', 'decoder_1/bahdanau_attention_1/dense_7/bias', 'decoder_1/bahdanau_attention_1/dense_8/kernel'] when minimizing the loss. If using `model.compile()`, did you forget to provide a `loss` argument?
  warnings.warn(


Epoch 1 Batch 0 Loss 1.9337
Epoch 1 Batch 100 Loss 1.0675
Epoch 1 Batch 200 Loss 0.9342
Epoch 1 Batch 300 Loss 0.8982
Epoch 1 Batch 400 Loss 0.8046
Epoch 1 Batch 500 Loss 0.8250
Epoch 1 Batch 600 Loss 0.7061
Saving checkpoint for epoch 1 at ./checkpoints/train/ckpt-24
Time taken for 1 epoch: 261.90 sec

Epoch 2 Batch 0 Loss 0.7763
Epoch 2 Batch 100 Loss 0.7827
Epoch 2 Batch 200 Loss 0.7609
Epoch 2 Batch 300 Loss 0.7033
Epoch 2 Batch 400 Loss 0.6784
Epoch 2 Batch 500 Loss 0.6345
Epoch 2 Batch 600 Loss 0.6147
Saving checkpoint for epoch 2 at ./checkpoints/train/ckpt-25
Time taken for 1 epoch: 253.78 sec

Epoch 3 Batch 0 Loss 0.7288
Epoch 3 Batch 100 Loss 0.6402
Epoch 3 Batch 200 Loss 0.6795
Epoch 3 Batch 300 Loss 0.5870
Epoch 3 Batch 400 Loss 0.5863
Epoch 3 Batch 500 Loss 0.5691
Epoch 3 Batch 600 Loss 0.5629
Saving checkpoint for epoch 3 at ./checkpoints/train/ckpt-26
Time taken for 1 epoch: 264.66 sec

Epoch 4 Batch 0 Loss 0.5982
Epoch 4 Batch 100 Loss 0.5818
Epoch 4 Batch 200 Loss 0.58

In [15]:
import pickle

# Save the tokenizer from Notebook 1
with open('tokenizer.pkl', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
print("Tokenizer saved successfully!")

Tokenizer saved successfully!
